# ⚛️ Notebook 01: From Everyday Scheduling Intuition to 3-Body *Vaisheshika* HUBO & Ising Spin Glasses
**PravahaQ (`v0.1.0`) — Core Mathematical Deep-Dive Series**

---

## 🟢 Layer 1: Plain-English Intuition (For Non-Physicists & Builders)

Imagine managing an **Edge-AI Micro-Cluster** (servers running LLM requests) powered by a **local Solar + Battery Microgrid**. Every half-second, you face a massive tug-of-war:
1. **If you pack coupled AI tasks onto the same server**, you save network cable delay, **but** the server heats up.
2. **If two heavy AI tasks run simultaneously while the power grid is in Eco-Cooling Mode**, a **3-way domino effect** occurs: the transformer phase saturates and the server thermally throttles.

### Why Standard 2-Body Quantum Models (QUBO) Fail
- **Standard QUBO** only looks at *pairs* of decisions ($x_i x_j$). To model a 3-way domino effect ($x_i x_j x_k$), legacy software adds fake "helper qubits" (ancillas) that waste up to 50% of the quantum processor.
- **PravahaQ's *Vaisheshika* Compiler** models **1-body ($L_i$), 2-body ($Q_{ij}$), and 3-body cubic ($T_{ijk}$) interactions natively**, and replaces wasteful inequality tally-mark qubits with an **asymmetric ski-slope wall (Montañez-Barrera Unbalanced Penalization)**.

---

## 🔵 Layer 2: Visual Compiler Pipeline & 3-Qubit Hand Calculation

```mermaid
flowchart LR
    A["Binary Decisions\nx_i ∈ {0, 1}"] --> B["Vaisheshika 3-Local HUBO\nL_i x_i + Q_ij x_i x_j + T_ijk x_i x_j x_k"]
    B --> C["Unbalanced Inequality Penalty\nΦ(g) = -λ_1 g(x) + λ_2 g(x)^2\n(ZERO Slack Qubits)"]
    C --> D["Affine Spin Projection\nx_i = (1 - Z_i) / 2"]
    D --> E["3-Local Ising Spin Glass\nH_C = C_0 I + ∑ h_i Z_i + ∑ J_ij Z_i Z_j + ∑ K_ijk Z_i Z_j Z_k"]
```

---

## 🟣 Layer 3: Rigorous Mathematical Formulation (Theorem 1)

Given a cubic pseudo-Boolean polynomial over $\mathbf{x} \in \{0, 1\}^N$:
$$E_{\text{HUBO}}(\mathbf{x}) = c_0 + \sum_{i=1}^{N} L_i x_i + \sum_{1 \le i < j \le N} Q_{ij} x_i x_j + \sum_{1 \le i < j < k \le N} T_{ijk} x_i x_j x_k$$

Applying the spectral projection $x_i \mapsto \frac{1 - \hat{Z}_i}{2}$ where $\hat{Z}_i |0\rangle = +1|0\rangle$ and $\hat{Z}_i |1\rangle = -1|1\rangle$ yields the exact diagonal **Spin-Glass Hamiltonian** $\hat{H}_C$:
$$\hat{H}_C = C_0 \hat{I} + \sum_{i=1}^{N} h_i \hat{Z}_i + \sum_{1 \le i < j \le N} J_{ij} \hat{Z}_i \hat{Z}_j + \sum_{1 \le i < j < k \le N} K_{ijk} \hat{Z}_i \hat{Z}_j \hat{Z}_k$$

with closed-form tensor transformations:
$$K_{ijk} = -\frac{1}{8} T_{ijk}, \qquad J_{ij} = \frac{1}{4} Q_{ij} + \frac{1}{8} \sum_{k \notin \{i,j\}} T_{\{i,j,k\}}, \qquad h_i = -\frac{1}{2} L_i - \frac{1}{4} \sum_{j \neq i} Q_{\{i,j\}} - \frac{1}{8} \sum_{j < k \neq i} T_{\{i,j,k\}}$$

In [1]:
import sys
from pathlib import Path

# Ensure local src/ is on Python path when running directly from notebooks/
sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

from pravaha_q.math_engine import HUBOPolynomial, hubo_to_ising, analyze_ising_spectrum

# 1. Construct the exact 3-qubit problem from docs/01_VISUAL_MATH_WALKTHROUGH.md:
#    E(x1, x2, x3) = -4*x1 + 2*x2 - 3*x3 - 2*x1*x2 + 2*x1*x3 + 8*x1*x2*x3
hubo_3q = HUBOPolynomial(num_variables=3, variable_labels=["Node1_Task", "Node2_Task", "Eco_Mode"])
hubo_3q.add_linear(0, -4.0)
hubo_3q.add_linear(1, 2.0)
hubo_3q.add_linear(2, -3.0)
hubo_3q.add_quadratic(0, 1, -2.0)
hubo_3q.add_quadratic(0, 2, 2.0)
hubo_3q.add_cubic(0, 1, 2, 8.0)

ising_3q = hubo_to_ising(hubo_3q)

print("=== 3-Qubit Worked Walkthrough Verification ===")
print(f"Ising Constant Offset C_0 : {ising_3q.constant_offset}")
print(f"1-Body Fields h_i         : {ising_3q.h_linear}")
print(f"2-Body Couplings J_ij     : {ising_3q.j_quadratic}")
print(f"3-Body Cubic K_ijk        : {ising_3q.k_cubic}\n")

print("Bitstring | Spins (z1,z2,z3) | E_HUBO(x) | <z|H_C|z> | Diff")
print("-" * 62)
for idx in range(8):
    bstr = format(idx, "03b")
    spins = tuple(1 - 2 * int(c) for c in bstr)
    e_h = hubo_3q.evaluate(bstr)
    e_i = ising_3q.evaluate_bitstring(bstr)
    tag = "  <-- GLOBAL OPTIMUM" if bstr == "101" else ("  <-- 3-BODY OVERLOAD!" if bstr == "111" else "")
    print(f"   {bstr}    |  ({spins[0]:+d}, {spins[1]:+d}, {spins[2]:+d})    |  {e_h:6.2f}   |  {e_i:6.2f}   | {abs(e_h - e_i):.2e}{tag}")

=== 3-Qubit Worked Walkthrough Verification ===
Ising Constant Offset C_0 : -1.5
1-Body Fields h_i         : {0: 1.0, 1: -1.5}
2-Body Couplings J_ij     : {(0, 1): 0.5, (0, 2): 1.5, (1, 2): 1.0}
3-Body Cubic K_ijk        : {(0, 1, 2): -1.0}

Bitstring | Spins (z1,z2,z3) | E_HUBO(x) | <z|H_C|z> | Diff
--------------------------------------------------------------
   000    |  (+1, +1, +1)    |    0.00   |    0.00   | 0.00e+00
   001    |  (+1, +1, -1)    |   -3.00   |   -3.00   | 0.00e+00
   010    |  (+1, -1, +1)    |    2.00   |    2.00   | 0.00e+00
   011    |  (+1, -1, -1)    |   -1.00   |   -1.00   | 0.00e+00
   100    |  (-1, +1, +1)    |   -4.00   |   -4.00   | 0.00e+00
   101    |  (-1, +1, -1)    |   -5.00   |   -5.00   | 0.00e+00  <-- GLOBAL OPTIMUM
   110    |  (-1, -1, +1)    |   -4.00   |   -4.00   | 0.00e+00
   111    |  (-1, -1, -1)    |    3.00   |    3.00   | 0.00e+00  <-- 3-BODY OVERLOAD!


## 2. Scaling Up: Compiling a Full Coupled Edge-AI + 3-Phase Microgrid Digital Twin

Now let's synthesize a realistic **2-Node, 3-Task, 2-Power-State ($N = 10$ qubit)** coupled infrastructure problem, inspect its **Unbalanced Inequality Penalty Curve**, and analyze its exact eigenspectrum ($\Delta E$ spectral gap and geometric frustration index).

In [2]:
from pravaha_q.domain import generate_coupled_problem
from pravaha_q.math_engine import VaisheshikaHUBOCompiler, evaluate_unbalanced_penalty_curve

problem = generate_coupled_problem(num_nodes=2, num_tasks=3, num_power_states=2, seed=42)
compiler = VaisheshikaHUBOCompiler(enable_cubic_terms=True)
hubo = compiler.compile(problem)
ising = hubo_to_ising(hubo)
spectrum = analyze_ising_spectrum(ising)

print(f"Problem Name              : {problem.name}")
print(f"Total Binary Variables (N): {problem.total_binary_variables} ({problem.num_routing_vars} Routing + {problem.num_power_vars} Power Dispatch)")
print(f"Compiled Ising Terms      : {len(ising.h_linear)} 1-Body | {len(ising.j_quadratic)} 2-Body | {len(ising.k_cubic)} 3-Body Cubic")
print(f"Exact Ground Energy E_0   : {spectrum.ground_energy:.4f}")
print(f"First Excited Energy E_1  : {spectrum.first_excited_energy:.4f} (Spectral Gap Delta_E = {spectrum.spectral_gap:.4f})")
print(f"Ground-State Bitstring z* : |{spectrum.ground_state_bitstring}>")
print(f"Spin-Glass Frustration    : {spectrum.frustration_index:.4f}")

Problem Name              : PravahaQ-2N-3T-2P-Seed42
Total Binary Variables (N): 10 (6 Routing + 4 Power Dispatch)
Compiled Ising Terms      : 10 1-Body | 15 2-Body | 6 3-Body Cubic
Exact Ground Energy E_0   : -28.9142
First Excited Energy E_1  : -27.4815 (Spectral Gap Delta_E = 1.4327)
Ground-State Bitstring z* : |1001100110>
Spin-Glass Frustration    : 0.2184
